In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import copy

import numpy as np 

from astropy.io import fits
from astropy.table import Table, Column

import matplotlib.pyplot as plt

### SUCD-1 of M104

- [An ultra-compact dwarf around the Sombrero galaxy (M104): the nearest massive UCD](https://ui.adsabs.harvard.edu/abs/2009MNRAS.394L..97H/abstract)
    - SUCD1 is not heavily dark matter dominated (e.g. Dabringhausen et al. 2008; Forbes et al. 2008), nor does it require an exotic IMF (Mieske & Kroupa 2008; Murray 2009), and it has not been strongly influenced by the host galaxy (Fellhauer & Kroupa 2006).
    - Detected in X-ray; X-ray luminosity is consistent with low-mass X-ray binary.
- Key Information:
    - RA = 12 40 03.13; Dec = -11 40 04.3 or (RA, Dec) = (190.0130417, -11.6678611) deg
    - M104 at distance of 9.0 Mpc
    - V=17.46 mag; B-V=0.91 mag; B-R=1.50 mag; V-R=0.58 mag
    - J=15.6 mag; H=14.9 mag; K=14.7 mag
    - $M_V = -12.31$ mag; $M_K=-15.1$ mag
    - $\sigma_0 = 31.0 \pm 6.9 {\rm km/s}$ 
    - $M/L_{K} =1.65$; $M_{\star} = 3.6 \times 10^7 M_{\odot}$
    - $M/L_{V} =4.36$; $M_{\star} = 3.0 \times 10^7 M_{\odot}$
    - $r_{\rm h} = 14.7 \pm 1.4$ pc
    - $M_{\rm virial} = 3.3 \pm 1.7 \times 10^{7} M_{\odot}$

- [Photometric relationships with other photometric systems](https://gea.esac.esa.int/archive/documentation/GDR2/Data_processing/chap_cu5pho/sec_cu5pho_calibr/ssec_cu5pho_PhotTransf.html)
    - **This will change for EDR3**
    - $(G - V) = -0.01746 + 0.008092 \times (V-I) - 0.2810 \times (V-I)^2 + 0.03655 \times (V-I)^3; \sigma=0.04670$
    - $(G - V) = -0.02269 + 0.01784 \times (V-R) - 1.016 \times (V-R)^2 + 0.2225 \times (V-R)^3; \sigma=0.04895$
    - $(G - V) = -0.02907 - 0.02385 \times (B-V) - 0.2297 \times (B-V)^2 - 0.001768 \times (B-V)^3; \sigma=0.04895$
    - $(G - K_s) = 0.6613 + 12.073 \times (H-K_s) - 1.359 \times (H-K_s)^2; \sigma=0.03692$

In [4]:
# Gaia G-band magnitude based on V-R color
17.46 - 0.02269 + 0.01784 * 0.58 -1.016 * (0.58 ** 2) + 0.2225 * (0.58 ** 3)

17.14928722

In [6]:
def v_to_gaia_g(v_mag, color, color_type='V-R'):
    """Convert Johnson V-band magnitude to Gaia G magnitude"""
    if color_type == 'V-R':
        G_mag = v_mag - 0.02269 + 0.01784 * color - 1.016 * (color ** 2) + 0.2225 * (color ** 3)
        G_sigma = 0.04895
    elif color_type == 'V-I':
        G_mag = v_mag - 0.01746 + 0.00809 * color - 0.281 * (color ** 2) + 0.0366 * (color ** 3)
        G_sigma = 0.04670
    elif color_type == 'B-V':
        G_mag = v_mag - 0.02907 - 0.02385 * color - 0.230 * (color ** 2) - 0.0018 * (color ** 3)
        G_sigma = 0.04895
    else:
        raise ValueError("Wrong color type!")
        
    return G_mag, G_sigma
    